In [ ]:
%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from ipywidgets import (
    FloatSlider,
    IntSlider,
    RadioButtons,
    HTML,
    Output,
    VBox,
    HBox,
    Layout
)

from IPython.display import (
    display,
    Math,
    clear_output
)

# ============================================================
# DIFFERENCE OPERATORS Δ AND E
# ============================================================

plt.ioff()

# ============================================================
# CLASSIC JUPYTER + JUPYTERLAB / NOTEBOOK 7 / BINDER
# ============================================================

display(HTML("""
<style>

.container {
    width: 98% !important;
    max-width: none !important;
}

.output_area,
.output_subarea {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.output_scroll {
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
    box-shadow: none !important;
}

.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow: visible !important;
    resize: none !important;
}

.jupyter-matplotlib::-webkit-resizer,
.jupyter-matplotlib-figure::-webkit-resizer {
    display: none !important;
}

.diff-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    gap: 18px !important;
    align-items: center !important;
}

.diff-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
}

.diff-radio > label {
    display: none !important;
}

.diff-title {
    font-family: Arial, sans-serif;
    font-size: 20px;
    font-weight: bold;
    color: #6f3fa0;
}

.diff-label {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
}

.diff-value {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
    color: #0b3d91;
}

</style>
"""))

# ============================================================
# SYMBOLIC VARIABLES
# ============================================================

n = sp.symbols(
    'n',
    integer=True,
    nonnegative=True
)

omega = sp.symbols(
    'omega',
    real=True
)

# ============================================================
# SEQUENCE SELECTOR
# ============================================================

sequence_selector = RadioButtons(
    options=[
        ('y[n] = n²', 'quadratic'),
        ('y[n] = 2ⁿ', 'exponential'),
        ('y[n] = sin(ωn)', 'sinusoidal')
    ],
    value='quadratic',
    description='',
    layout=Layout(
        width='560px'
    )
)

sequence_selector.add_class(
    'diff-radio'
)

# ============================================================
# SLIDER SETTINGS
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='220px'
)

label_layout = Layout(
    width='140px',
    min_width='140px'
)

value_layout = Layout(
    width='60px',
    min_width='60px',
    margin='0px 0px 0px 6px'
)

row_layout = Layout(
    width='450px',
    height='40px',
    min_height='40px',
    align_items='center'
)

# ============================================================
# ANGULAR FREQUENCY SLIDER
# ============================================================

omega_slider = FloatSlider(
    min=0.10,
    max=np.pi,
    step=0.05,
    value=np.pi / 4,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

omega_label = HTML(
    '<div class="diff-label">Angular freq. ω:</div>',
    layout=label_layout
)

omega_value = HTML(
    '<div class="diff-value">0.785</div>',
    layout=value_layout
)

omega_row = HBox(
    [
        omega_label,
        omega_slider,
        omega_value
    ],
    layout=row_layout
)

# ============================================================
# MAXIMUM VALUE OF n
#
# n ranges from 0 to 12.
# Therefore, for y[n] = 2^n the largest possible value is
#
# 2^12 = 4096.
# ============================================================

N_slider = IntSlider(
    min=5,
    max=12,
    step=1,
    value=12,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

N_label = HTML(
    '<div class="diff-label">Maximum n:</div>',
    layout=label_layout
)

N_value = HTML(
    '<div class="diff-value">12</div>',
    layout=value_layout
)

N_row = HBox(
    [
        N_label,
        N_slider,
        N_value
    ],
    layout=row_layout
)

# ============================================================
# SEQUENCE PARAMETERS PANEL
# ============================================================

parameters_panel = VBox(
    [
        HTML("""
        <div class="diff-title" style="margin-bottom:10px;">
            Sequence Parameters
        </div>
        """),

        HTML("""
        <div style="
            font-family:Arial;
            font-size:14px;
            font-weight:bold;
            margin-bottom:5px;
        ">
            Select sequence:
        </div>
        """),

        sequence_selector,
        omega_row,
        N_row
    ],
    layout=Layout(
        width='480px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# SYMBOLIC RESULTS
# ============================================================

selected_sequence_output = Output(
    layout=Layout(
        width='350px'
    )
)

operator_output = Output(
    layout=Layout(
        width='820px'
    )
)

# ============================================================
# SELECTED SEQUENCE:
# LABEL + EQUATION ON THE SAME LINE
# ============================================================

selected_sequence_label = HTML(
    """
    <div style="
        font-family:Arial, sans-serif;
        font-size:14px;
        white-space:nowrap;
        padding-top:4px;
    ">
        Selected sequence:
    </div>
    """,
    layout=Layout(
        width='125px',
        min_width='125px'
    )
)

selected_sequence_row = HBox(
    [
        selected_sequence_label,
        selected_sequence_output
    ],
    layout=Layout(
        width='820px',
        align_items='center',
        gap='8px',
        overflow='visible'
    )
)

symbolic_panel = VBox(
    [
        HTML("""
        <div class="diff-title" style="margin-bottom:8px;">
            Symbolic Results
        </div>
        """),

        selected_sequence_row,
        operator_output
    ],
    layout=Layout(
        width='850px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# TOP ROW
# ============================================================

top_row = HBox(
    [
        parameters_panel,
        symbolic_panel
    ],
    layout=Layout(
        width='1360px',
        gap='16px',
        align_items='stretch',
        overflow='visible'
    )
)

# ============================================================
# SYMBOLIC SEQUENCE
# ============================================================

def get_symbolic_sequence(mode):

    if mode == 'quadratic':

        return n**2

    elif mode == 'exponential':

        return 2**n

    else:

        return sp.sin(
            omega * n
        )

# ============================================================
# SYMBOLIC OPERATORS
# ============================================================

def calculate_symbolic_operators(y):

    # --------------------------------------------------------
    # E y[n] = y[n+1]
    # --------------------------------------------------------

    E1 = sp.simplify(
        y.subs(
            n,
            n + 1
        )
    )

    # --------------------------------------------------------
    # E² y[n] = y[n+2]
    # --------------------------------------------------------

    E2 = sp.simplify(
        y.subs(
            n,
            n + 2
        )
    )

    # --------------------------------------------------------
    # Δ y[n] = y[n+1] - y[n]
    # --------------------------------------------------------

    D1 = sp.trigsimp(
        sp.expand_trig(
            sp.simplify(
                E1 - y
            )
        )
    )

    # --------------------------------------------------------
    # Δ² y[n]
    # --------------------------------------------------------

    D2 = sp.trigsimp(
        sp.expand_trig(
            sp.simplify(
                E2
                -
                2 * E1
                +
                y
            )
        )
    )

    return (
        E1,
        E2,
        D1,
        D2
    )

# ============================================================
# CURRENT SYMBOLIC EXPRESSIONS
# ============================================================

current_y_symbolic = None
current_E1_symbolic = None
current_E2_symbolic = None
current_D1_symbolic = None
current_D2_symbolic = None

# ============================================================
# FIGURE CREATION FUNCTION
#
# ALL THREE FIGURES HAVE EXACTLY THE SAME SIZE
# ============================================================

def create_figure(
    title,
    ylabel,
    title_color
):

    fig, ax = plt.subplots(
        figsize=(4.25, 3.75)
    )

    fig.canvas.header_visible = False
    fig.canvas.footer_visible = False
    fig.canvas.toolbar_visible = False

    fig.canvas.layout = Layout(
        width='425px',
        height='375px',
        overflow='visible'
    )

    ax.set_title(
        title,
        fontsize=13,
        fontweight='bold',
        color=title_color
    )

    ax.set_xlabel(
        'n',
        fontsize=10
    )

    ax.set_ylabel(
        ylabel,
        fontsize=10
    )

    ax.grid(
        True,
        linestyle=':',
        alpha=0.40
    )

    line, = ax.plot(
        [],
        [],
        marker='o',
        markersize=4,
        linewidth=1.4
    )

    fig.subplots_adjust(
        left=0.15,
        right=0.97,
        top=0.88,
        bottom=0.15
    )

    return (
        fig,
        ax,
        line
    )

# ============================================================
# FIGURES
# ============================================================

fig_y, ax_y, line_y = create_figure(
    'Original Sequence y[n]',
    'y[n]',
    '#6f3fa0'
)

fig_d1, ax_d1, line_d1 = create_figure(
    'First Difference Δy[n]',
    'Δy[n]',
    '#0b3d91'
)

fig_d2, ax_d2, line_d2 = create_figure(
    'Second Difference Δ²y[n]',
    'Δ²y[n]',
    '#16802b'
)

# ============================================================
# FIXED X RANGE
#
# Slider changes do NOT change the axes.
# ============================================================

for ax in (
    ax_y,
    ax_d1,
    ax_d2
):

    ax.set_xlim(
        -0.5,
        12.5
    )

# ============================================================
# FIXED Y RANGES FOR EACH SEQUENCE TYPE
#
# These ranges change only when another sequence is selected.
# They do NOT change while the sliders are moving.
# ============================================================

def set_axes_for_sequence(mode):

    # --------------------------------------------------------
    # y[n] = n²
    #
    # maximum n = 12
    # maximum y = 144
    # maximum Δy = 25
    # Δ²y = 2
    # --------------------------------------------------------

    if mode == 'quadratic':

        ax_y.set_ylim(
            -10,
            160
        )

        ax_d1.set_ylim(
            -2,
            28
        )

        ax_d2.set_ylim(
            1.0,
            3.0
        )

    # --------------------------------------------------------
    # y[n] = 2^n
    #
    # maximum y[12] = 4096
    #
    # Δy[n] = 2^n
    # Δ²y[n] = 2^n
    # --------------------------------------------------------

    elif mode == 'exponential':

        ax_y.set_ylim(
            -150,
            4400
        )

        ax_d1.set_ylim(
            -150,
            4400
        )

        ax_d2.set_ylim(
            -150,
            4400
        )

    # --------------------------------------------------------
    # y[n] = sin(ωn)
    # --------------------------------------------------------

    else:

        ax_y.set_ylim(
            -1.2,
            1.2
        )

        ax_d1.set_ylim(
            -2.2,
            2.2
        )

        ax_d2.set_ylim(
            -4.2,
            4.2
        )

# ============================================================
# FIGURES ROW
# ============================================================

figures_row = HBox(
    [
        fig_y.canvas,
        fig_d1.canvas,
        fig_d2.canvas
    ],
    layout=Layout(
        width='1320px',
        gap='15px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='visible'
    )
)

# ============================================================
# NUMERICAL VALUES PANEL
# ============================================================

table_html = HTML()

numerical_panel = VBox(
    [
        HTML("""
        <div class="diff-title" style="margin-bottom:8px;">
            Numerical Values (first 10 samples)
        </div>
        """),

        table_html
    ],
    layout=Layout(
        width='920px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        margin='8px 0px 0px 0px',
        overflow='visible'
    )
)

# ============================================================
# UPDATE SYMBOLIC RESULTS
#
# IMPORTANT:
# Called ONLY when the selected sequence changes.
#
# Slider movement therefore does NOT call clear_output().
# ============================================================

def update_symbolic_results(change=None):

    global current_y_symbolic
    global current_E1_symbolic
    global current_E2_symbolic
    global current_D1_symbolic
    global current_D2_symbolic

    mode = sequence_selector.value

    current_y_symbolic = get_symbolic_sequence(
        mode
    )

    (
        current_E1_symbolic,
        current_E2_symbolic,
        current_D1_symbolic,
        current_D2_symbolic
    ) = calculate_symbolic_operators(
        current_y_symbolic
    )

    # ========================================================
    # SELECTED SEQUENCE
    # ========================================================

    with selected_sequence_output:

        clear_output(
            wait=True
        )

        display(
            Math(
                r'\displaystyle y[n]='
                +
                sp.latex(
                    current_y_symbolic
                )
            )
        )

    # ========================================================
    # OPERATOR EXPRESSIONS
    # ========================================================

    with operator_output:

        clear_output(
            wait=True
        )

        display(
            Math(
                r'\displaystyle '
                r'Ey[n]'
                r'='
                r'y[n+1]'
                r'='
                +
                sp.latex(
                    current_E1_symbolic
                )
            )
        )

        display(
            Math(
                r'\displaystyle '
                r'E^2y[n]'
                r'='
                r'y[n+2]'
                r'='
                +
                sp.latex(
                    current_E2_symbolic
                )
            )
        )

        display(
            Math(
                r'\displaystyle '
                r'\Delta y[n]'
                r'='
                r'y[n+1]-y[n]'
                r'='
                +
                sp.latex(
                    current_D1_symbolic
                )
            )
        )

        # ----------------------------------------------------
        # SECOND DIFFERENCE ON ITS OWN LINE
        # ----------------------------------------------------

        display(
            Math(
                r'\displaystyle '
                r'\Delta^2y[n]'
                r'='
                r'y[n+2]-2y[n+1]+y[n]'
                r'='
                +
                sp.latex(
                    current_D2_symbolic
                )
            )
        )

# ============================================================
# UPDATE NUMERICAL RESULTS
#
# Slider movement updates only:
#
# - curve data
# - numerical values
# - table
#
# It does NOT recreate figures.
# It does NOT change the axis ranges.
# ============================================================

def update_numerical_results(change=None):

    mode = sequence_selector.value

    omega_value_current = (
        omega_slider.value
    )

    N_max = (
        N_slider.value
    )

    # ========================================================
    # ENABLE / DISABLE ω
    # ========================================================

    omega_slider.disabled = (
        mode != 'sinusoidal'
    )

    # ========================================================
    # VALUE LABELS
    # ========================================================

    omega_value.value = (
        f'<div class="diff-value">'
        f'{omega_value_current:.3f}'
        f'</div>'
    )

    N_value.value = (
        f'<div class="diff-value">'
        f'{N_max}'
        f'</div>'
    )

    # ========================================================
    # SUBSTITUTE NUMERICAL ω
    # ========================================================

    if mode == 'sinusoidal':

        y_numeric_expr = (
            current_y_symbolic.subs(
                omega,
                omega_value_current
            )
        )

        D1_numeric_expr = (
            current_D1_symbolic.subs(
                omega,
                omega_value_current
            )
        )

        D2_numeric_expr = (
            current_D2_symbolic.subs(
                omega,
                omega_value_current
            )
        )

    else:

        y_numeric_expr = (
            current_y_symbolic
        )

        D1_numeric_expr = (
            current_D1_symbolic
        )

        D2_numeric_expr = (
            current_D2_symbolic
        )

    # ========================================================
    # NUMERICAL FUNCTIONS
    # ========================================================

    y_function = sp.lambdify(
        n,
        y_numeric_expr,
        modules='numpy'
    )

    D1_function = sp.lambdify(
        n,
        D1_numeric_expr,
        modules='numpy'
    )

    D2_function = sp.lambdify(
        n,
        D2_numeric_expr,
        modules='numpy'
    )

    # --------------------------------------------------------
    # Include n = N_max.
    #
    # For N_max = 12:
    # n = 0,1,...,12
    # --------------------------------------------------------

    n_values = np.arange(
        0,
        N_max + 1
    )

    y_values = np.asarray(
        y_function(
            n_values
        ),
        dtype=float
    )

    D1_values = np.asarray(
        D1_function(
            n_values
        ),
        dtype=float
    )

    D2_values = np.asarray(
        D2_function(
            n_values
        ),
        dtype=float
    )

    # ========================================================
    # HANDLE CONSTANT EXPRESSIONS
    # ========================================================

    if y_values.ndim == 0:

        y_values = np.full(
            len(n_values),
            float(y_values)
        )

    if D1_values.ndim == 0:

        D1_values = np.full(
            len(n_values),
            float(D1_values)
        )

    if D2_values.ndim == 0:

        D2_values = np.full(
            len(n_values),
            float(D2_values)
        )

    # ========================================================
    # UPDATE EXISTING CURVES ONLY
    # ========================================================

    line_y.set_data(
        n_values,
        y_values
    )

    line_d1.set_data(
        n_values,
        D1_values
    )

    line_d2.set_data(
        n_values,
        D2_values
    )

    # ========================================================
    # NUMERICAL TABLE
    # ========================================================

    table_N = min(
        10,
        len(n_values)
    )

    rows = ""

    for k in range(
        table_N
    ):

        rows += f"""
        <tr>

            <td style="
                padding:6px 18px;
                border-bottom:1px solid #eeeeee;
                text-align:center;
            ">
                {n_values[k]}
            </td>

            <td style="
                padding:6px 18px;
                border-bottom:1px solid #eeeeee;
                text-align:center;
            ">
                {y_values[k]:.5f}
            </td>

            <td style="
                padding:6px 18px;
                border-bottom:1px solid #eeeeee;
                text-align:center;
            ">
                {D1_values[k]:.5f}
            </td>

            <td style="
                padding:6px 18px;
                border-bottom:1px solid #eeeeee;
                text-align:center;
            ">
                {D2_values[k]:.5f}
            </td>

        </tr>
        """

    table_html.value = f"""
    <table style="
        width:760px;
        border-collapse:collapse;
        font-family:Arial, sans-serif;
        font-size:13px;
    ">

        <tr style="
            background:#f4eff8;
            font-weight:bold;
        ">

            <td style="
                width:120px;
                padding:7px 18px;
                border:1px solid #d5cddd;
                text-align:center;
            ">
                n
            </td>

            <td style="
                width:180px;
                padding:7px 18px;
                border:1px solid #d5cddd;
                text-align:center;
            ">
                y[n]
            </td>

            <td style="
                width:180px;
                padding:7px 18px;
                border:1px solid #d5cddd;
                text-align:center;
            ">
                Δy[n]
            </td>

            <td style="
                width:180px;
                padding:7px 18px;
                border:1px solid #d5cddd;
                text-align:center;
            ">
                Δ²y[n]
            </td>

        </tr>

        {rows}

    </table>
    """

    # ========================================================
    # REDRAW ONLY
    # ========================================================

    fig_y.canvas.draw_idle()
    fig_d1.canvas.draw_idle()
    fig_d2.canvas.draw_idle()

# ============================================================
# SEQUENCE CHANGE
#
# The axis ranges may change ONLY here because the mathematical
# type of sequence has changed.
# ============================================================

def sequence_changed(change=None):

    mode = sequence_selector.value

    update_symbolic_results()

    set_axes_for_sequence(
        mode
    )

    update_numerical_results()

# ============================================================
# CONNECT CONTROLS
# ============================================================

sequence_selector.observe(
    sequence_changed,
    names='value'
)

# ------------------------------------------------------------
# CONTINUOUS SLIDER UPDATE
#
# No clear_output()
# No figure recreation
# No axis rescaling
# ------------------------------------------------------------

omega_slider.observe(
    update_numerical_results,
    names='value'
)

N_slider.observe(
    update_numerical_results,
    names='value'
)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_symbolic_results()

set_axes_for_sequence(
    sequence_selector.value
)

update_numerical_results()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        top_row,
        figures_row,
        numerical_panel
    ],
    layout=Layout(
        width='1380px',
        gap='12px',
        overflow='visible'
    )
)

display(
    main_layout
)